# minatar_endpoint

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from analysis.curves import episode_average, return_curve, subsample
from analysis.stats import mean_ci, min_max_normalize
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Each setting's component per agent, and each figure set's agents in drawing
# order with their colors. Endpoint comes last so it is drawn on top. Agents
# with no stored runs yet are left out of every plot.
COMPONENTS = {
    "medium": {
        "Large": "ddqn_minatar",
        "Small": "ddqn_medium_minatar",
        "Unanchored": "unanchored_minatar",
        "Endpoint": "endpoint_minatar",
    },
    "small": {
        "Large": "ddqn_minatar",
        "Small": "ddqn_small_minatar",
        "Unanchored": "unanchored_small_minatar",
        "Endpoint": "endpoint_small_minatar",
    },
}
FIGURES = {
    "poster": {
        "Large": "black",
        "Small": "tab:blue",
        "Unanchored": "#a90064",
        "Endpoint": "#009f16",
    },
}
# Legend and bar order.
ORDER = ["Endpoint", "Large", "Small", "Unanchored"]
# Memory relative to the 100k large buffer.
TITLES = {"medium": "MinAtar 10x Smaller", "small": "MinAtar 50x Smaller"}

component = EXPERIMENT.component("ddqn_minatar")
GAMES = component.sweep["ENV_HYPERS.GAME"]

In [ ]:
# Load each component's stored runs per game once. Compute its mean return per
# game with a 95% bootstrap CI, and each run's episode-average return for the
# bar plots.
names = {name for agents in COMPONENTS.values() for name in agents.values()}
curves = {}
episode_scores = {}
for name in names:
    for game in GAMES:
        runs = load_results(EXPERIMENT, name, where={"ENV_HYPERS.GAME": game})
        if runs is None:
            continue
        curve = return_curve(runs.reward, runs.done)
        timesteps, returns = subsample(curve)
        curves[name, game] = (timesteps, mean_ci(returns), len(returns))
        episode_scores[name, game] = episode_average(runs.reward, runs.done)


def present(figure, setting):
    agents = COMPONENTS[setting]
    return [
        agent
        for agent in FIGURES[figure]
        if any((agents[agent], game) in curves for game in GAMES)
    ]

In [ ]:
# Style for the all-games grids, laid out as in minatar_deterministic: one row
# with a panel per game, each with its own y-axis. Every piece of text shares
# one size.
TOTAL_STEPS = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS
FONT_SIZE = 32
GRID_STYLE = {
    "figure.figsize": (32, 8),
    "font.family": "Times New Roman",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
BAND_ALPHA = 0.2
# The panels share colors, so only the left one carries the legend.
GRID_LEGEND = {
    "loc": "lower right",
    "bbox_to_anchor": (1.04, -0.04),
    "handlelength": 1,
    "handletextpad": 0.4,
}
GRID_LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 14}
PANEL_ASPECT = 1
# Ticks sit only at the axis ends, so the labels can tuck in between them.
LABEL_PAD = -20
GRID_XLABEL = "Time Steps"
GRID_XTICKS = [0, TOTAL_STEPS]
GRID_XTICK_LABELS = ["0", f"{TOTAL_STEPS // 1_000_000}M"]
GRID_YLABEL = "Return"
YTICK_STEP = 10


def title(game):
    return game.replace("_", " ").title()


def top_of_axis(highs):
    # The axis ends at the next multiple of YTICK_STEP above every curve drawn.
    return int(np.ceil(np.nanmax(highs) / YTICK_STEP) * YTICK_STEP)

In [ ]:
# Plot every game in one grid per figure set and setting, with the legend in
# the first panel.
grids = {}
for figure, colors in FIGURES.items():
    for setting, agents in COMPONENTS.items():
        shown = present(figure, setting)
        if not shown:
            continue
        with plt.rc_context(GRID_STYLE):
            grid, axes = plt.subplots(1, len(GAMES))
            for ax, game in zip(axes, GAMES, strict=True):
                ax.set_box_aspect(PANEL_ASPECT)
                drawn = [agent for agent in shown if (agents[agent], game) in curves]
                if not drawn:
                    ax.axis("off")
                    continue
                highs = []
                for agent in drawn:
                    steps, (mean, low, high), _ = curves[agents[agent], game]
                    color = colors[agent]
                    ax.fill_between(steps, low, high, color=color, alpha=BAND_ALPHA)
                    ax.plot(steps, mean, color=color)
                    highs.append(np.nanmax(high))
                top = top_of_axis(highs)
                ax.set_title(title(game))
                ax.set_xlabel(GRID_XLABEL, labelpad=LABEL_PAD)
                ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
                ax.set_yticks([0, top], ["0", str(top)])
                ax.set_ylim(0, top)
                ax.set_ylabel(GRID_YLABEL, labelpad=LABEL_PAD)
            grid.tight_layout()
            labels = [agent for agent in ORDER if agent in shown]
            handles = [
                Line2D([], [], color=colors[agent], **GRID_LEGEND_MARKER)
                for agent in labels
            ]
            axes[0].legend(handles, labels, **GRID_LEGEND)
        grids[f"grid_{setting}"] = grid
plt.show()

In [ ]:
# Style for the normalized-score bar plots.
BAR_STYLE = {
    "figure.figsize": (14, 7),
    "font.family": "Times New Roman",
    "font.size": 32,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 32,
    "axes.labelsize": 32,
    "ytick.labelsize": 32,
}
BAR_WIDTH = 0.6  # a quarter thinner than the default 0.8
BAR_YLABEL = "Normalized Average Return"
BAR_YLABEL_PAD = 0
BAR_ASPECT = 1
# A panel's y-axis top, which is also its upper tick, is the next tenth above
# its tallest bar.
BAR_TOP_STEP = 0.1
# Bars are named inside, reading up from their base; a bar too short for that
# carries its name on top instead, one word per line.
BAR_INSIDE_LABEL = {"rotation": 90, "ha": "center", "va": "bottom", "color": "white"}
BAR_INSIDE_MIN = 0.3
# Per-figure overrides: bar order, and figures whose names always sit inside.
BAR_ORDERS = {"poster": ["Large", "Small", "Unanchored", "Endpoint"]}
# Poster names all sit inside; a smaller font fits "Unanchored" in its short bar.
BAR_ALWAYS_INSIDE = {"poster"}
BAR_LABEL_SIZES = {"Unanchored": 24}
BAR_TOP_LABEL = {"ha": "center", "va": "bottom"}
BAR_LABEL_GAP = 0.01
# A name that would run past the top of its bar shrinks to fit, leaving this much
# of the bar's height free.
BAR_FIT_MARGIN = 0.95

In [ ]:
# Normalize each game's runs onto [0, 1] with bounds shared by a panel's agents,
# then average over games and seeds.
def normalized_means(figure, setting):
    agents = COMPONENTS[setting]
    order = BAR_ORDERS.get(figure, ORDER)
    shown = [agent for agent in order if agent in present(figure, setting)]
    complete = [
        game
        for game in GAMES
        if all((agents[agent], game) in episode_scores for agent in shown)
    ]
    if not complete:
        return {}
    totals = dict.fromkeys(shown, 0.0)
    for game in complete:
        stacks = [episode_scores[agents[agent], game] for agent in shown]
        for agent, stack in zip(shown, min_max_normalize(stacks), strict=True):
            totals[agent] += float(np.mean(stack))
    return {agent: total / len(complete) for agent, total in totals.items()}


bar_figures = {}
for figure, colors in FIGURES.items():
    panels = {setting: normalized_means(figure, setting) for setting in COMPONENTS}
    if not any(panels.values()):
        continue
    with plt.rc_context(BAR_STYLE):
        fig, axes = plt.subplots(1, len(panels))
        inside = []
        for column, (ax, (setting, means)) in enumerate(
            zip(axes, panels.items(), strict=True)
        ):
            ax.set_box_aspect(BAR_ASPECT)
            for position, (agent, mean) in enumerate(means.items()):
                ax.bar(position, mean, width=BAR_WIDTH, color=colors[agent])
                if figure in BAR_ALWAYS_INSIDE or mean >= BAR_INSIDE_MIN:
                    size = BAR_LABEL_SIZES.get(agent)
                    name = (position, BAR_LABEL_GAP, agent)
                    text = ax.text(*name, fontsize=size, **BAR_INSIDE_LABEL)
                    inside.append((ax, text, mean))
                else:
                    label = agent.replace(" ", "\n")
                    top_label = (position, mean + BAR_LABEL_GAP, label)
                    ax.text(*top_label, **BAR_TOP_LABEL)
            ax.set_title(TITLES[setting])
            ax.set_xticks([])
            top = round(np.ceil(max(means.values()) / BAR_TOP_STEP) * BAR_TOP_STEP, 1)
            ax.set_yticks([0, top])
            ax.set_ylim(0, top)
            if column == 0:
                ax.set_ylabel(BAR_YLABEL, labelpad=BAR_YLABEL_PAD)
        fig.tight_layout()
        fig.canvas.draw()
        for ax, text, mean in inside:
            start, top = (
                ax.transData.transform((0, y))[1] for y in (BAR_LABEL_GAP, mean)
            )
            room = BAR_FIT_MARGIN * (top - start)
            needed = text.get_window_extent().height
            if needed > room:
                text.set_fontsize(text.get_fontsize() * room / needed)
    bar_figures["bars"] = fig
plt.show()

In [ ]:
# Save every figure as a PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in {**grids, **bar_figures}.items():
    for suffix in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{suffix}", bbox_inches="tight")
print(f"saved {len(grids) + len(bar_figures)} figure(s) to {PLOTS_DIR}")